# Horizontal-flip feature extraction — development splits only

Run this notebook first. It uses the existing RGB, AVG, and NTSC v3 CNN checkpoints without fitting or changing weights. Each image is normalized exactly as in the corresponding v3 extraction notebook. For each complete ordered split, it calls `extractor.predict(..., batch_size=128)` twice on the original view and requires the two float32 output arrays to be exactly equal. It then predicts the width-axis flip once and saves `(original + flipped) / 2` as float32. The complete split is passed to each call, so the same Keras data adapter handles the final partial batch (64 training rows and 16 validation rows).

The historical v3 embeddings are compared across all 40K training and 10K validation images. Maximum, mean, percentile, violating-value, and affected-image statistics are retained as reporting-only diagnostics; historical tolerance violations do not reject repeatable current embeddings. Extraction still stops for repeatability failures, nonfinite values, changed dimensions, split/index/label mismatches, preprocessing mismatches, or checkpoint/provenance conflicts. The legacy v3 archives do not contain CNN hashes, so historical extraction checkpoint identity cannot be proven from those archives.

This notebook reads only cached CIFAR-10 training batches. It never loads `test_batch`, fits a model, runs SVM optimization, or modifies a CNN checkpoint. New archives are written under the versioned experiment directory and existing archives are never overwritten.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "flip_features.py").is_file():
    ROOT = ROOT / "tugas-7"
if not (ROOT / "flip_features.py").is_file():
    raise FileNotFoundError("Run from the repository root or tugas-7")
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
EXPERIMENT_ID = "fusion_flip_v1_001"
from flip_features import (
    ABSOLUTE_TOLERANCES, RTOL, VERIFICATION_PROTOCOL,
    extract_representation, feature_path,
)

BATCH_SIZE = 128  # Required to reproduce the original v3 extraction path.
RUN_RGB_EXTRACTION = False
RUN_AVG_EXTRACTION = False
RUN_NTSC_EXTRACTION = True

print("Experiment:", EXPERIMENT_ID)
print("Verification protocol:", VERIFICATION_PROTOCOL)
print("Historical comparison tolerances (reporting only):", {
    "atol": ABSOLUTE_TOLERANCES, "rtol": RTOL,
})
for name in ("RGB", "AVG", "NTSC"):
    print(name, "->", feature_path(ROOT, EXPERIMENT_ID, name))


Experiment: fusion_flip_v1_001
Verification protocol: same-session-exact-repeatability-v1
Historical comparison tolerances (reporting only): {'atol': {'RGB': 0.0005, 'AVG': 0.0005, 'NTSC': 0.0006}, 'rtol': 0.0001}
RGB -> /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/feature-engineering/experiments/fusion_flip_v1_001/rgb_flip_features.npz
AVG -> /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/feature-engineering/experiments/fusion_flip_v1_001/avg_flip_features.npz
NTSC -> /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-7/feature-engineering/experiments/fusion_flip_v1_001/ntsc_flip_features.npz


## Extract RGB, then AVG, then NTSC

Enable one switch at a time and execute this cell. Each completed archive is verified when reloaded. A matching existing RGB or AVG archive is verified and skipped without requiring re-extraction. Keep every switch disabled during code review.


In [2]:
for representation, enabled in (
    ("RGB", RUN_RGB_EXTRACTION),
    ("AVG", RUN_AVG_EXTRACTION),
    ("NTSC", RUN_NTSC_EXTRACTION),
):
    if enabled:
        print("Starting:", representation)
        print(extract_representation(
            ROOT, EXPERIMENT_ID, representation, batch_size=BATCH_SIZE,
        ))
    else:
        print("Pending:", representation)


Pending: RGB
Pending: AVG
Starting: NTSC


I0000 00:00:1791424037.173104   41977 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791424037.213014   41977 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791424038.177312   41977 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791424039.157860   41977 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3672 MB mem

  5/313 ━━━━━━━━━━━━━━━━━━━━ 9s 32ms/step

I0000 00:00:1791424046.788228   42050 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


313/313 ━━━━━━━━━━━━━━━━━━━━ 18s 40ms/step
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step
NTSC train repeatability: {"affected_images": 0, "criterion": "two float32 predict outputs must be exactly equal for every feature value", "differing_feature_values": 0, "exact_match": true, "feature_values": 20480000, "images": 40000, "max_absolute_difference": 0.0}
NTSC train historical v3 comparison (reporting only): {"affected_image_offsets": [], "affected_image_offsets_truncated": false, "affected_images": 0, "atol": 0.0006, "feature_values": 20480000, "images": 40000, "largest_violation_examples": [], "max_absolute_difference": 0.0005603432655334473, "mean_absolute_difference": 7.706272299401462e-06, "percentile_absolute_difference": {"100.0": 0.0005603432655334473, "50.0": 1.9073486328125e-06, "90.0": 2.2470951080322266e-05, "95.0": 3.427267074584961e-05, "99.0": 6.687641143798828e-05, "99.9": 0.00012412667274475098}, "rtol": 0.0001, "violating_feature_values": 0, "within_provisional_tolerance":

## Next

After all three new archives exist, run **Feature Engineering Flip Combination.ipynb**. Do not run the old v3 Combination notebook, which writes a different archive format and includes official test features.
